## Decision Tree accuracy experimnet

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV,cross_val_predict

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,balanced_accuracy_score,roc_auc_score,average_precision_score,log_loss,brier_score_loss

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean_hr_job_change.csv"

EXPERIMENT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "decision_tree_accuracy"
)

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

ID_COLUMN = "enrollee_id"
TARGET_COLUMN = "target"

X = df.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (16284, 14)
Test: (2874, 14)


### Baseline Decision Tree

In [2]:
numeric_features = [
    "city_development_index",
    "training_hours",
    "experience_missing",
    "experience_numeric",
    "last_new_job_missing",
    "last_new_job_numeric"
]

categorical_features = [
    "city",
    "gender",
    "relevent_experience",
    "enrolled_university",
    "education_level",
    "major_discipline",
    "company_size",
    "company_type"
]

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_features),
        ("cat", categorical_preprocessor, categorical_features)
    ]
)

In [3]:
baseline_tree = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=7,
            min_samples_leaf=5,
            min_samples_split=20,
            class_weight=None,
            random_state=RANDOM_STATE
        ))
    ]
)

baseline_tree.fit(X_train, y_train)
baseline_prob = baseline_tree.predict_proba(X_test)[:, 1]
baseline_pred = (baseline_prob >= 0.31).astype(int)

In [4]:
baseline_results = {
    "Model": "Current Decision Tree",
    "Threshold": 0.31,
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "F1": f1_score(y_test, baseline_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, baseline_pred),
    "ROC-AUC": roc_auc_score(y_test, baseline_prob),
    "PR-AUC": average_precision_score(y_test, baseline_prob),
    "Log Loss": log_loss(y_test, baseline_prob),
    "Brier Score": brier_score_loss(y_test, baseline_prob)
}

pd.DataFrame([baseline_results])

,Model,Threshold,Accuracy,Precision,Recall,F1,Balanced Accuracy,ROC-AUC,PR-AUC,Log Loss,Brier Score
0,Current Decision Tree,0.31,0.786708,0.554507,0.737796,0.633154,0.770382,0.803015,0.534296,0.506655,0.139861


### Prepareing Grid Search to find better model w.r.t. accuracy

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ))
    ]
)

In [6]:
tree_accuracy_grid = {
    "model__max_depth": [3, 5, 7, 10, 12, 15, 20, None],
    "model__min_samples_split": [2, 5, 10, 20, 50, 100],
    "model__min_samples_leaf": [1, 2, 5, 10, 20, 50],
    "model__max_features": [None,"sqrt","log2"],
    "model__class_weight": [None,"balanced"]
}

In [7]:
tree_accuracy_search = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=tree_accuracy_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

tree_accuracy_search.fit(X_train, y_train)

print("Best CV Accuracy:")
print(tree_accuracy_search.best_score_)

print("\nBest Parameters:")
print(tree_accuracy_search.best_params_)

Fitting 5 folds for each of 1728 candidates, totalling 8640 fits
Best CV Accuracy:
0.7939073769669409

Best Parameters:
{'model__class_weight': None, 'model__max_depth': 7, 'model__max_features': None, 'model__min_samples_leaf': 5, 'model__min_samples_split': 100}


In [8]:
tuned_tree = tree_accuracy_search.best_estimator_
tuned_prob = tuned_tree.predict_proba(X_test)[:, 1]
tuned_pred = (tuned_prob >= 0.50).astype(int)

tuned_results = {
    "Model": "Tuned Decision Tree",
    "Threshold": 0.50,
    "Accuracy": accuracy_score(y_test, tuned_pred),
    "Precision": precision_score(y_test, tuned_pred),
    "Recall": recall_score(y_test, tuned_pred),
    "F1": f1_score(y_test, tuned_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, tuned_pred),
    "ROC-AUC": roc_auc_score(y_test, tuned_prob),
    "PR-AUC": average_precision_score(y_test, tuned_prob),
    "Log Loss": log_loss(y_test, tuned_prob),
    "Brier Score": brier_score_loss(y_test, tuned_prob)
}

pd.DataFrame([baseline_results,tuned_results])

,Model,Threshold,Accuracy,Precision,Recall,F1,Balanced Accuracy,ROC-AUC,PR-AUC,Log Loss,Brier Score
0,Current Decision Tree,0.31,0.786708,0.554507,0.737796,0.633154,0.770382,0.803015,0.534296,0.506655,0.139861
1,Tuned Decision Tree,0.50,0.790536,0.568047,0.669456,0.614597,0.750120,0.808789,0.540093,0.445140,0.138571


In [9]:
import pickle
import json

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
model_path = EXPERIMENT_DIR / "decision_tree_accuracy_optimized.pkl"

with open(model_path, "wb") as f:
    pickle.dump(tree_accuracy_search.best_estimator_, f)

params_path = EXPERIMENT_DIR / "best_parameters.json"
with open(params_path, "w") as f:
    json.dump(tree_accuracy_search.best_params_, f, indent=4)

cv_summary = pd.DataFrame([{
    "experiment": "Decision Tree Accuracy Optimization",
    "best_cv_accuracy": tree_accuracy_search.best_score_,
    "best_parameters": str(tree_accuracy_search.best_params_)
}])

cv_summary.to_csv(
    EXPERIMENT_DIR / "cv_summary.csv",
    index=False
)

print("Experiment model saved to:")
print(model_path)
print("\nExperiment artifacts:")
for file in sorted(EXPERIMENT_DIR.iterdir()):
    print(" -", file.name)

Experiment model saved to:
e:\hr-job-change-ml\artifacts\experiments\decision_tree_accuracy\decision_tree_accuracy_optimized.pkl

Experiment artifacts:
 - best_parameters.json
 - cv_summary.csv
 - decision_tree_accuracy_optimized.pkl


In [10]:
with open(model_path, "rb") as f:
    loaded_experiment_tree = pickle.load(f)

print(type(loaded_experiment_tree))

<class 'sklearn.pipeline.Pipeline'>
